<a id="contents"></a>
# MODELING UTILITY FUNCTIONS

- [**GENERAL FUNCTIONS**](#gen_utilities)
- [**EVALUATION FUNCTIONS**](#eval_functions)
- [**VISUALIZATION FUNCTIONS**](#visual_functions)
- [**DATA CLEANING FUNCTIONS**](#data_cleaning)
- [**CROSS VALIDATON FUNCTIONS**](#cross_val)
- [**MODELING FUNCTIONS**](#model_fun)
    - [**GENERAL MODELING FUNCTIONS**](#gen_model_fun)
    - [**LINEAR REGRESSION FUNCTIONS**](#lin_reg)
    - [**RANDOM FOREST FUNCTIONS**](#r_forest)
    - [**XGBOOST FUNCTIONS**](#xgboost)
    - [**Light GBM FUNCTIONS**](#lgbm)

In [2]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import r2_score, mean_squared_error

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.model_selection import RandomizedSearchCV

from sklearn.ensemble import RandomForestRegressor

from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet

from sklearn.feature_selection import VarianceThreshold

from sklearn.feature_selection import f_classif
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.base import clone

from lightgbm import LGBMRegressor
from merf import MERF

from sklearn.utils.parallel import Parallel, delayed

In [3]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')
warnings.filterwarnings('ignore', category=UserWarning, module='sklearn.utils.parallel')

# Ignore all of the sklearn warnings.
warnings.filterwarnings('ignore', module='sklearn')

if sys.platform == 'linux':
    !chmod 644 ~/.local/share/jupyter/history.sqlite
    #!rm ~/.local/share/jupyter/history.sqlite

import logging
logging.getLogger("merf").setLevel(logging.WARNING)

chmod: /Users/kosaraju_b/.local/share/jupyter/history.sqlite: No such file or directory


<a id="gen_utilities"></a>
# GENERAL FUNCTIONS

In [1]:
GREEN_TICK = "\u2705"
RED_CROSS  = "\u274C"
WARNING    = "\u26A0\uFE0F"

In [4]:
# IDENTIFY LOW AND HIGH AGB PLOTS
# Classifies all 59 plots into three groups:
#  Group-1: Near-zero variance plots whose R² scores are mathematically
#           unreliable as holdout sets
#  Group-2: High-AGB plots that are statistical outliers at either
#           the dataset level or within their own site,
#  Group-3: Everything else.
#
#Returns both plot-level and site-level categories for use in grouped CV evaluation.
def get_low_and_high_agb_plots(y, plot_groups):
    near_zero_plots, high_agb_plots = get_plot_categories(y, plot_groups)
    
    within_site_high_agb = get_within_site_high_agb_plots(y, plot_groups, near_zero_plots)
    high_agb_plots = list(set(high_agb_plots + within_site_high_agb))
    
    # Derive site-level categories from plot-level results
    near_zero_sites = list(set(
        plot.rsplit('_', maxsplit=1)[0]
        for plot in near_zero_plots
    ))
    
    # Exclude near-zero sites from high-AGB sites
    # Big Creek_3 is high-AGB at plot level but Big Creek as a site
    # is near-zero — site level takes precedence here
    high_agb_sites = list(set(
        plot.rsplit('_', maxsplit=1)[0]
        for plot in high_agb_plots
        if plot.rsplit('_', maxsplit=1)[0] not in near_zero_sites
    ))
    
    print("\nNear-zero sites:", sorted(near_zero_sites))
    print("High-AGB sites :", sorted(high_agb_sites))
    return near_zero_sites, high_agb_sites, near_zero_plots, high_agb_plots

In [5]:
# IDENTIFY LOW AND HIGH AGB SITES (SITE-LEVEL GROUPED CV)
# - Same classification logic as get_low_and_high_agb_plots but operates at site
#   level rather than plot level.
# - Used when grouped CV holds out entire sites.
# - Near-zero sites are detected via the largest relative gap in log variance
#   across sites.
# - High-AGB sites are detected via IQR on site-level max AGB.
def get_site_categories(y, groups):
    site_groups = groups.map(lambda x: x.rsplit('_', maxsplit=1)[0])
    site_stats  = y.groupby(site_groups).agg(
        max_agb = 'max',
        var_agb = 'var',
        n_rows  = 'count'
    )

    # ── High-AGB sites: IQR on site-level max AGB ─────────────────
    max_agb_values = site_stats['max_agb']
    Q1             = max_agb_values.quantile(0.25)
    Q3             = max_agb_values.quantile(0.75)
    IQR            = Q3 - Q1
    high_threshold = Q3 + 1.5 * IQR
    high_agb_sites = site_stats[max_agb_values > high_threshold].index.tolist()

    # ── Near-zero sites: largest relative gap in log variance ──────
    log_var         = np.log1p(y).groupby(site_groups).var()
    log_var_sorted  = log_var.sort_values()
    relative_gaps   = log_var_sorted.diff() / log_var_sorted.shift(1)
    largest_gap_idx = relative_gaps.idxmax()
    gap_position    = log_var_sorted.index.get_loc(largest_gap_idx)
    near_zero_threshold = log_var_sorted.iloc[gap_position - 1]
    near_zero_sites = log_var[log_var <= near_zero_threshold].index.tolist()

    print(f"High-AGB threshold  (Q3 + 1.5*IQR)       : {high_threshold:.2f} kg")
    print(f"Near-zero threshold (largest relative gap): {near_zero_threshold:.6f}")

    print(f"\nNear-zero variance sites:")
    for site in near_zero_sites:
        print(f"  {site:20s} : log var = {log_var[site]:.6f}")

    print(f"\nHigh-AGB sites:")
    for site in high_agb_sites:
        print(f"  {site:20s} : max AGB = {site_stats.loc[site, 'max_agb']:.1f} kg")

    return near_zero_sites, high_agb_sites

In [6]:
# IDENTIFY LOW AND HIGH AGB PLOTS (DATASET-LEVEL)
# Classifies plots using two data-driven thresholds.
#  - High-AGB plots are those those max AGB exceeds Q3 + 1.5*IQR
#    across all 59 plots — capturing Channel Caye and New River plots
#    as global outliers.
#  - Near-zero plots are those below the largest relative jump in 
#    log variance across plots — capturing Frenchman Caye, Shipstern, 
#    and Big Creek plots where R² is mathematically unreliable.
def get_plot_categories(y, groups):
    plot_stats = y.groupby(groups).agg(max_agb = 'max',
                                       var_agb = 'var',
                                       n_rows  = 'count')

    # High-AGB plots: IQR on plot-level max AGB
    max_agb_values = plot_stats['max_agb']
    Q1             = max_agb_values.quantile(0.25)
    Q3             = max_agb_values.quantile(0.75)
    IQR            = Q3 - Q1
    high_threshold = Q3 + 1.5 * IQR
    high_agb_plots = plot_stats[
        max_agb_values > high_threshold
    ].index.tolist()

    # Near-zero plots: largest relative gap in log variance
    log_var         = np.log1p(y).groupby(groups).var()
    log_var_sorted  = log_var.sort_values()
    relative_gaps   = log_var_sorted.diff() / log_var_sorted.shift(1)
    largest_gap_idx = relative_gaps.idxmax()
    gap_position    = log_var_sorted.index.get_loc(largest_gap_idx)
    near_zero_threshold = log_var_sorted.iloc[gap_position - 1]
    near_zero_plots = log_var[log_var < near_zero_threshold].index.tolist()

    print(f"High-AGB threshold  : {high_threshold:.2f} kg")
    print(f"Near-zero threshold : {near_zero_threshold:.6f}")

    print(f"\nNear-zero variance plots:")
    for plot in near_zero_plots:
        print(f"  {plot:25s} : log var = {log_var[plot]:.6f}")

    print(f"\nHigh-AGB plots:")
    for plot in high_agb_plots:
        print(f"  {plot:25s} : max AGB = {plot_stats.loc[plot, 'max_agb']:.1f} kg")

    return near_zero_plots, high_agb_plots

In [7]:
# IDENTIFY HIGH AGB PLOTS WITHIN EACH SITE
#  - Finds plots that are AGB outliers relative to other plots in the same site.
#  - These are missed by the dataset-level IQR because their AGB is not extreme
#    globally — only within their site.
#  - Example: Big Creek_3 (max 12.6 kg) is not a global outlier but is well above
#    the other Big Creek plots (max 6-8 kg).
#  - Near-zero plots are excluded before the within-site IQR is computed to prevent
#    sites like Big Creek from being skipped entirely due to their near-zero plots.
def get_within_site_high_agb_plots(y, plot_groups,
                                   near_zero_plots,
                                   iqr_multiplier=1.5):
    site_groups    = plot_groups.map(lambda x: x.rsplit('_', maxsplit=1)[0])
    near_zero_set  = set(near_zero_plots)
    print("\nwithin-site high-AGB plots")
    within_site_outliers = []
    for site in site_groups.unique():
        site_mask = (site_groups == site).values              # ← only change
        plot_max  = y[site_mask].groupby(plot_groups[site_mask]).max()
        plot_max  = plot_max[~plot_max.index.isin(near_zero_set)]
        if len(plot_max) < 3:
            continue
        Q1             = plot_max.quantile(0.25)
        Q3             = plot_max.quantile(0.75)
        IQR            = Q3 - Q1
        high_threshold = Q3 + iqr_multiplier * IQR
        outlier_plots  = plot_max[plot_max > high_threshold].index.tolist()
        if outlier_plots:
            print(f"{site:20s} : within-site high-AGB plots = {outlier_plots}")
            within_site_outliers.extend(outlier_plots)
    return within_site_outliers

In [8]:
def split_data_by_groups(X_var, y_var, groups):

    if len(groups.unique()) < 2:
        return split_data_quantile(X_var, y_var)
    
    X_local = X_var.copy(deep=True)
    y_local = y_var.copy(deep=True)

    # Let us say there are 5 unique plots and 15 trees in total, aka, 3 trees per plot.
    #  Plot A — rows 0, 1, 2
    #  Plot B — rows 3, 4, 5
    #  Plot C — rows 6, 7, 8
    #  Plot D — rows 9, 10, 11
    #  Plot E — rows 12, 13, 14
    #
    # GroupShuffleSplit
    # -----------------
    # This is a cross-validation iterator that generates train/test indices
    # to split data into subsets, ensuring specific groups do not overlap
    # between sets.
    # 
    # n_splits=1 => One fixed split. 80-20 split in this case.
    # OUTPUT of GroupShuffleSplit(n_splits=1..)
    #   train_idx = [0, 1, 2, 3, 4, 5, 9, 10, 11, 12, 13, 14] # plots A,B,D,E
    #   test_idx  = [6, 7, 8]                                 # plot C only
    #
    # next(gss.split())
    # -----------------
    # gss.split() is a generator that would produce n_splits pairs.
    # Since n_splits=1 there is only one pair to produce.
    # The next() call retrieves that one pair and returns it.
    # OUTPUT of next(gss.split())
    #    train_idx = [0,1,2,3,4,5,9,10,11,12,13,14]
    #    test_idx  = [6,7,8]
    #
    # Interpolation (The "Cheat" Split)
    # ---------------------------------
    #  If the training set has 8 trees from Plot_A and the test set has the
    #  remaining 2 trees from Plot_A, the  model is interpolating. It already
    #  knows the specific "vibe" of Plot_A.
    #
    #  Result:
    #   - High R2 that is false.
    #   - The model is just identifying trees it already "knows" by proxy.
    #
    # Extrapolation (The "True" Split)
    # ---------------------------------
    #  If you put every single tree from Plot_A into the test set and train the model
    #  only on trees from Plot_B, C, and D, the model is extrapolating.
    #
    #  The "New Site": In this context, Plot_A is the "new site."
    #  It is a geographic location the model has never seen.
    #
    #  Result:
    #    - A lower, but realistic R2.
    #    - This tells you how well your model will perform if you take it to a
    #      completely different part of Panama or Brazil.
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

    train_idx, test_idx = next(gss.split(X_local, y_local, groups=groups))
    
    X_train = X_local.iloc[train_idx]
    X_test  = X_local.iloc[test_idx]
    y_train = y_local.iloc[train_idx]
    y_test  = y_local.iloc[test_idx]

    train_plots = set(groups.iloc[train_idx]) #plot_id
    test_plots  = set(groups.iloc[test_idx])
    
    overlap     = train_plots & test_plots
    #print(f"Train plots : {len(train_plots)}")
    #print(f"Test plots  : {len(test_plots)}")
    #print(f"Overlapping plots: {len(overlap)}")  # must be 0
    assert not len(overlap)
    
    return X_train, X_test, y_train, y_test

In [9]:
def split_data_quantile(X_var, y_var):
    # The train_test_split() splits randomly by rows — it
    # does not know about any groups (e.g., plot ids, etc).
    # 
    # When the data is split with this method, trees from the same plot
    # can end up in both train and test. The model trains on 58 trees
    # from plot X and is tested on the remaining 7 trees from the same plot.
    # This is pseudo-replication.
    #
    # pseudo-replication
    # ------------------
    # Pseudo-replication occurs when individual data points are treated as
    # independent observations in a statistical model, even though they are
    # actually clustered or correlated due to a shared environment.

    # When is pseudo-replication NOT useful?
    #  - Pseudo-replication is a problem specifically when your data has a
    #    hierarchical structure, e.g., trees nested within plots,
    #    students nested within schools, etc.
    #  - In those cases observations within the same group are correlated and
    #   random splitting leaks information.
    # E.g.
    #  - When trees from the same plot appear in both train and test,
    #    the model is evaluated on data it has effectively already seen,
    #    i.e., same EMIT pixel, same site conditions, same species mix.
    #  - The R² will be inflated.
    #  - You report a number that does not reflect real world performance on new sites.

    # When is pseudo-replication useful?
    #  - Pseudo-replication is acceptable when your dataset has no meaningful
    #    grouping structure, aka, the observations are independent of each other.

    y_quantiles = pd.qcut(y_var, q=4, labels=False)
    X_train, X_test, y_train, y_test = train_test_split(X_var,
                                                        y_var,
                                                        test_size=0.2,
                                                        random_state=42,
                                                        stratify=y_quantiles)

    return X_train, X_test, y_train, y_test

In [10]:
def split_data_plot_stratified(X, y, plot_groups, test_size=0.2, random_state=42):
    """
    Train/test split that ensures every plot contributes to both train and test.
    Within each plot, test_size fraction of rows go to test.
    """
    train_idx = []
    test_idx  = []

    for plot in plot_groups.unique():
        plot_mask = (plot_groups == plot)
        idx       = plot_groups[plot_mask].index.tolist()

        if len(idx) < 2:
            # too small to split — put in train
            train_idx.extend(idx)
            continue

        n_test = max(1, int(len(idx) * test_size))
        rng    = np.random.RandomState(random_state)
        test   = rng.choice(idx, size=n_test, replace=False).tolist()
        train  = [i for i in idx if i not in test]

        train_idx.extend(train)
        test_idx.extend(test)

    X_train = X.loc[train_idx]
    X_test  = X.loc[test_idx]
    y_train = y.loc[train_idx]
    y_test  = y.loc[test_idx]

    train_plots = plot_groups.loc[train_idx]
    test_plots  = plot_groups.loc[test_idx]

    print(f"Train : {len(X_train)} rows, {train_plots.nunique()} plots")
    print(f"Test  : {len(X_test)} rows,  {test_plots.nunique()} plots")
    print(f"\nMax AGB in train : {y_train.max():.1f} kg")
    print(f"Max AGB in test  : {y_test.max():.1f} kg")

    return X_train, X_test, y_train, y_test

In [11]:
# Leave-One-Group-Out data splitter.
# Generates one (X_train, X_test, y_train, y_test) tuple per unique group,
# each time holding out all rows belonging to one group as the test set
# and using the remaining groups for training. No group overlaps between
# train and test in any fold.
#
# Analogy to split_data_by_groups:
#  - split_data_by_groups     : one fixed 80/20 split, one test group
#  - split_data_by_groups_logo: N splits, each group gets one turn as test       
def split_data_by_groups_logo(X_var, y_var, groups):
    logo   = LeaveOneGroupOut()
    splits = []
    for train_idx, test_idx in logo.split(X_var, y_var, groups=groups):
        X_train = X_var.iloc[train_idx]
        X_test  = X_var.iloc[test_idx]
        y_train = y_var.iloc[train_idx]
        y_test  = y_var.iloc[test_idx]
        splits.append((X_train, X_test, y_train, y_test))
    return splits

<a id="eval_functions"></a>
# EVALUATION FUNCTIONS

In [13]:
def get_site_from_plot(plot_name):
    return plot_name.rsplit('_', maxsplit=1)[0]

<a id="visual_functions"></a>
# VISUALIZATION FUNCTIONS

In [17]:
# Plots alpha vs R² for regularised models (Ridge, Lasso, ElasticNet)
# or a learning curve for plain linear regression.
def plot_linear_reg_metrics(model_type, model,
                            X_train_scaled, y_train_log,
                            label):
    from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
    from sklearn.model_selection import cross_val_score, learning_curve

    if model_type in ('ridge', 'lasso', 'elasticnet'):

        if model_type == 'ridge':
            alphas = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5,
                      1.0, 5.0, 10.0, 50.0, 100.0, 500.0]
        else:
            alphas = [0.0001, 0.0005, 0.001, 0.005, 0.01,
                      0.05, 0.1, 0.5, 1.0, 5.0]

        train_r2 = []
        cv_r2    = []

        for alpha in alphas:
            if model_type == 'ridge':
                m = Ridge(alpha=alpha)
            elif model_type == 'lasso':
                m = Lasso(alpha=alpha, max_iter=10000)
            else:
                best_l1 = model.l1_ratio_ if hasattr(model, 'l1_ratio_') else 0.5
                m = ElasticNet(alpha=alpha, l1_ratio=best_l1, max_iter=10000)

            m.fit(X_train_scaled, y_train_log)
            train_r2.append(m.score(X_train_scaled, y_train_log))
            cv_r2.append(cross_val_score(m, X_train_scaled, y_train_log,
                                         cv=5, scoring='r2').mean())

        best_alpha = getattr(model, 'alpha_', None)

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.semilogx(alphas, train_r2, color='steelblue',
                    linewidth=1.5, label='Train R²')
        ax.semilogx(alphas, cv_r2, color='darkorange',
                    linewidth=1.5, label='CV R² (5-fold)')
        if best_alpha is not None:
            ax.axvline(x=best_alpha, color='red', linestyle='--',
                       linewidth=1, label=f'Selected alpha: {best_alpha:.4f}')
        ax.set_xlabel("Alpha  (regularization strength →  more regularization)")
        ax.set_ylabel("R²")
        ax.set_title(f"{model_type.upper()} — Alpha vs R²  |  {label}")
        ax.legend()
        plt.tight_layout()
        plt.show()
        print(f"Selected alpha : {best_alpha}")

    else:
        # Plain linear regression — learning curve
        train_sizes, train_scores, val_scores = learning_curve(
            LinearRegression(),
            X_train_scaled, y_train_log,
            cv=5, scoring='r2',
            train_sizes=np.linspace(0.1, 1.0, 10),
            n_jobs=-1
        )
        train_mean = train_scores.mean(axis=1)
        val_mean   = val_scores.mean(axis=1)

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(train_sizes, train_mean, color='steelblue',
                linewidth=1.5, label='Train R²')
        ax.plot(train_sizes, val_mean, color='darkorange',
                linewidth=1.5, label='CV R² (5-fold)')
        ax.set_xlabel("Training set size")
        ax.set_ylabel("R²")
        ax.set_title(f"Linear Regression — Learning curve  |  {label}")
        ax.legend()
        plt.tight_layout()
        plt.show()

In [18]:
def plot_correlation_matrix(X, y, top_n=20, figsize=(12, 8)):
    df_corr = X.copy()
    df_corr['plant_AGB_kg'] = y.values

    target_corr = df_corr.corr()['plant_AGB_kg'].drop('plant_AGB_kg').abs().sort_values(ascending=False)
    
    # Compute correlation of all features against target only
    target_corr = df_corr.corr()['plant_AGB_kg'].drop('plant_AGB_kg')
    target_corr = target_corr.abs().sort_values(ascending=False).head(top_n)

    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    target_corr.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'Top {top_n} Feature Correlations with plant_AGB_kg')
    ax.set_xlabel('Feature')
    ax.set_ylabel('Absolute Correlation')
    ax.axhline(y=0.1, color='red', linestyle='--', label='threshold=0.1')
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    return target_corr

In [19]:
def residual_analysis(y_pred, residuals):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    axes[0].scatter(y_pred, residuals, alpha=0.4)
    axes[0].axhline(0, color='red', linestyle='--')
    axes[0].set_xlabel('Predicted AGB')
    axes[0].set_ylabel('Residuals')
    axes[0].set_title('Residuals vs Predicted')

    axes[1].hist(residuals, bins=30)
    axes[1].set_xlabel('Residual')
    axes[1].set_title('Residual Distribution')
    
    stats.probplot(residuals, dist="norm", plot=axes[2])
    axes[2].set_title('QQ Plot')

    plt.tight_layout()
    plt.show()

In [20]:
def show_importances(results, N=10):
    importances = results["importances"].sort_values(ascending=False)
    
    # Feature importances
    print(f"\nTop {N} feature importances:")
    for feat, imp in importances.head(N).items():
        bar = '=' * int(imp * 50)
        print(f"  {feat:45s} {imp:.4f}  {bar}")

<a id="data_cleaning"></a>
# DATA CLEANING FUNCTIONS

In [22]:
def get_categorical_cols(X_data):
    return X_data.select_dtypes(include=['object', 'category']).columns.tolist()

def get_numerical_cols(X_data):
    return X_data.select_dtypes(include=['number']).columns.tolist()

In [23]:
def irrelevant_categorical_features(X, y, alpha=0.05, min_samples=2):
    """
    Calls categorical_matters for every categorical column.
    Drops columns that do not significantly affect the target.
    Returns filtered X and summary dataframe.
    """
    X_filtered       = X.copy()
    categorical_cols = get_categorical_cols(X)
    
    results      = []
    cols_to_drop = []

    for col in categorical_cols:
        print(f"\n--- {col} ---")
        relevance = categorical_relevance(X, y,
                                          col,
                                          alpha=alpha,
                                          min_samples=min_samples)

        results.append({'feature': col, 'relevant?': relevance})

        if not relevance:
            cols_to_drop.append(col)

    summary = pd.DataFrame(results)
    return cols_to_drop, summary

In [24]:
def remove_low_variance_cols(X_data,
                             exclude_cols,
                             threshold,
                             exclude_categorical=True):
    if exclude_categorical:
        categorical_cols = X_data.select_dtypes(include=['object', 'category']).columns.tolist()
        continuous_cols = [c for c in X_data.columns if c not in categorical_cols and c not in exclude_cols]
    else:
        continuous_cols = [c for c in X_data.columns if c not in exclude_cols]
    
    # Apply variance threshold
    X_continuous = X_data[continuous_cols]
    selector     = VarianceThreshold(threshold=threshold)
    selector.fit(X_continuous)
    
    low_variance_cols = X_continuous.columns[~selector.get_support()].tolist()
    print(f"Total low variance columns removed: {len(low_variance_cols)}")
    
    kept_continuous = X_continuous.columns[selector.get_support()].tolist()
    
    # Reassemble — keep all encoded columns, keep only high-variance continuous columns
    features = None
    if exclude_categorical:
        features = kept_continuous + categorical_cols + exclude_cols
    else:
        features = kept_continuous + exclude_cols
    
    print(f"Features after variance filtering: {len(X_data.columns)}")
    return X_data[features], low_variance_cols

In [25]:
def remove_uncorrelated_numerical_cols(X_data, y_data,
                                       threshold,
                                       exclude_cols,
                                       debug=False):
    categorical_cols  = X_data.select_dtypes(
                            include=['object', 'category']
                        ).columns.tolist()
    continuous_cols   = [c for c in X_data.columns
                         if c not in categorical_cols
                         and c not in exclude_cols]
    X_continuous      = X_data[continuous_cols]

    # Drop zero-variance columns before computing correlation
    # corrwith divides by std — zero std produces NaN and RuntimeWarning
    zero_var_cols     = X_continuous.columns[X_continuous.var() == 0].tolist()
    if zero_var_cols:
        print(f"Zero variance columns skipped: {zero_var_cols}")
    X_continuous      = X_continuous.drop(columns=zero_var_cols)

    correlations      = X_continuous.corrwith(y_data).abs()\
                                    .sort_values(ascending=False)
    strong_corr_cols  = correlations[correlations >= threshold].index.tolist()
    weak_corr_cols    = correlations[correlations <  threshold].index.tolist()

    if debug:
        print(f"Strong correlations kept : {strong_corr_cols}")
    print(f"Weak correlations removed: {weak_corr_cols}")

    X_data = X_data[strong_corr_cols + categorical_cols + exclude_cols]
    print(f"\nTotal features after correlation filtering: {len(X_data.columns)}")
    return X_data

In [26]:
from scipy.stats import f_oneway

def categorical_relevance(X, y, categorical_col, alpha=0.05, min_samples=2):
    # Get unique categories
    categories = X[categorical_col].unique()

    # Group target values by category.
    
    # Also, filter out groups with insufficient samples
    # Why?
    # ----
    #  ANOVA requires at least 2 samples per group to compute within-group variance.
    #  If any species category has only 1 tree — or zero trees in a particular subset
    #  of the data, the F-statistic cannot be computed and returns NaN.
    
    groups = [y[X[categorical_col] == cat].values for cat in categories
              if len(y[X[categorical_col] == cat]) >= min_samples]
    if len(groups) < 2:
        print(f"{categorical_col} : skipped — fewer than 2 valid groups after filtering")
        return False
    
    # One-way ANOVA
    f_stat, p_value = f_oneway(*groups)
    
    # What F-statistic Means
    # ----------------------
    # F = variance between species groups / variance within species groups
    # High F
    #  E.g.
    #   — AGB varies more between species than within species.
    #   - Species identity is a meaningful predictor of AGB.
    # Low F (close to 1)
    #  E.g.
    #   — AGB varies as much within species as between species.
    #   - Species identity adds little predictive value.

    # What does p-value Means?
    # -------------------
    #  It answers what is the probability of observing this F-statistic by chance
    #  if species had no effect on AGB?
    # p < 0.05   → species has a statistically significant effect on AGB
    # p > 0.05   → cannot conclude species affects AGB

    relevant = p_value < alpha
    
    print(f"Variable  : {categorical_col}")
    print(f"F-stat    : {f_stat:.4f}")
    print(f"p-value   : {p_value:.4f}")
    print(f"Relevant? : {relevant}")
    
    return relevant

def remove_uncorrelated_categorical_cols(X_data, y_data):
    cols_to_drop, summary = irrelevant_categorical_features(X_data, y_data)
    print(f"\n{summary}")
    print("\n")
    print(f"Strong correlations kept   : {len(X_data.columns) - len(cols_to_drop)}")
    print(f"Weak correlations removed: {len(cols_to_drop)}")

    return X_data.drop(columns=cols_to_drop)

In [27]:
def handle_null_columns(X_data):
    null_rows = X_data[X_data.isnull().any(axis=1)]
    total_nulls = X_data.isnull().sum().sum()
    
    print(f"Total NULL count           : {total_nulls}")
    print(f"Rows with at least one NULL: {len(null_rows)}")
    print(f"Total rows                 : {len(X_data)}")
    print(f"Percentage                 : {len(null_rows)/len(X_data)*100:.1f}%")
    
    # NULL count per column for only the affected rows
    null_col_counts = null_rows.isnull().sum().sort_values(ascending=False)
    
    print("\nNULL count per column in affected rows:")
    print(null_col_counts[null_col_counts > 0])

    vapor_bands = null_rows.isnull().sum()
    vapor_bands = vapor_bands[vapor_bands > 0].index.tolist()
    
    print(f"Dropping {len(vapor_bands)} columns:")
    print(vapor_bands)
    
    X_data = X_data.drop(columns=vapor_bands)
    print(f"\nNULL count after dropping: {X_data.isnull().sum().sum()}")

    return X_data

In [28]:
def remove_low_variance_plots(df, target_col='plant_AGB_kg',
                              plot_col='plot_id', threshold=None):
    
    plot_var = df.groupby(plot_col)[target_col].var()
    
    if threshold is None:
        threshold = plot_var.quantile(0.25)
        print(f"Auto threshold (Q25): {threshold:.4f}")
    
    low_var_plots = plot_var[plot_var <= threshold].index.tolist()
    print(f"Removing {len(low_var_plots)} plots: {low_var_plots}")
    
    df_clean = df[~df[plot_col].isin(low_var_plots)].copy()
    print(f"Rows: {len(df)} → {len(df_clean)}")
    print(f"Plots: {df[plot_col].nunique()} → {df_clean[plot_col].nunique()}")
    
    return df_clean

<a id="cross_val"></a>
# CROSS VALIDATON FUNCTIONS

In [30]:
def get_fold_sites(X, y_log, plot_groups, n_splits=10):
    n_splits   = min(n_splits, plot_groups.nunique())
    gkf        = GroupKFold(n_splits=n_splits)
    fold_sites = []
    fold_plots = []  # full composition per fold

    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y_log, plot_groups)
    ):
        plots_in_fold = plot_groups.iloc[test_idx].unique().tolist()
        # Primary plot — first non-near-zero, non-trivial plot
        primary = plots_in_fold[0]
        fold_sites.append(primary)
        fold_plots.append(plots_in_fold)

    return fold_sites, fold_plots

In [31]:
def cross_validate(model_func, X_data, y_data, cv=10, scoring='r2', display=True):
    cv_scores = cross_val_score(model_func,
                                X_data, y_data,
                                cv=cv,
                                scoring=scoring)
    if display:
        print("\n Cross-validation ---")
        print(f"CV R² mean: {cv_scores.mean():.4f}")
        print(f"CV R² std : {cv_scores.std():.4f}")
        print(f"CV scores : {cv_scores.round(3)}")

    return cv_scores.mean(), cv_scores.std(), cv_scores

In [32]:
def cross_validate_by_groups(model_func, X_data, y_data,
                             groups, n_splits=10, scoring='r2',
                             display=True):
    # Cap n_splits to the number of unique groups
    n_unique_groups = len(groups.unique())

    # Example:
    # n_unique_groups = 60
    #  GroupKFold(n_splits=60) => 1 group held out per fold  (leave-one-group-out)
    #  GroupKFold(n_splits=10) => 6 groups held out per fold
    #  GroupKFold(n_splits=6)  => 10 groups held out per fold
    #  GroupKFold(n_splits=2)  => 30 groups held out per fold
    n_splits = min(n_splits, n_unique_groups) # num folds <= group count

    # GroupKFold splits data into K folds ensuring all rows belonging
    # to the same group are entirely in one fold.
    # Example:
    #   Num plots = 5, n_splits = 5
    #   GroupKFold assigns each plot to exactly one fold:
    #    Fold 1 — test: Plot A    train: Plots B, C, D, E
    #    Fold 2 — test: Plot B    train: Plots A, C, D, E
    #    Fold 3 — test: Plot C    train: Plots A, B, D, E
    #    Fold 4 — test: Plot D    train: Plots A, B, C, E
    #    Fold 5 — test: Plot E    train: Plots A, B, C, D
    gkf       = GroupKFold(n_splits=n_splits)
    cv_scores = cross_val_score(model_func,
                                X_data, y_data,
                                cv=gkf.split(X_data, y_data, groups),
                                scoring=scoring)

    if display:
        print("\nGrouped Cross-validation ---")
        print(f"Grouped CV R² mean: {cv_scores.mean():.4f}")
        print(f"Grouped CV R² std : {cv_scores.std():.4f}")
        print(f"Grouped CV scores : {cv_scores.round(3)}")

    return cv_scores.mean(), cv_scores.std(), cv_scores

In [33]:
def select_best_model(experiments):
    max_val = max(v["cv_r2_mean"] for v in experiments.values())
    
    rf_best_labels = [
        label
        for label, vals in experiments.items()
        if vals["cv_r2_mean"] == max_val
    ]
    
    print(rf_best_labels)
    return rf_best_labels[0]

<a id="model_fun"></a>
# MODELING FUNCTIONS

<a id="gen_model_fun"></a>
## GENERAL MODELING FUNCTIONS

In [36]:
def post_processing(model,
                    X_train, X_test,
                    y_train, y_test, y_train_log,
                    cv, cv_func, groups,
                    label,
                    display=True):
    # Test set performance
    y_pred_log       = model.predict(X_test)
    y_train_pred_log = model.predict(X_train)
    
    # Inverse log
    y_pred       = np.expm1(y_pred_log)
    y_train_pred = np.expm1(y_train_pred_log)
    
    test_r2    = r2_score(y_test, y_pred)
    test_rmse  = np.sqrt(mean_squared_error(y_test, y_pred))
    
    train_r2   = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    
    residuals  = y_test - y_pred

    if display:
        print(f"\n {label}")
        print(f"Test R²     : {test_r2:.4f}")
        print(f"Test RMSE   : {test_rmse:.2f} kg")
        print(f"Train R² (log scale): {r2_score(y_train_log, y_train_pred_log):.4f}")
        print(f"Train R² (orig scale): {train_r2:.4f}")
        print(f"Train RMSE  : {train_rmse:.2f} kg")
        print(f"Num rows    : {X_train.shape[0]}")
        print(f"Num Features: {X_train.shape[1]}")

    # Cross-validation to confirm that the R^2 above is not by coincidence.
    cv_r2_mean, cv_r2_std, cv_scores = cross_validate(cv_func,
                                                      X_train,
                                                      y_train_log,
                                                      cv=cv,
                                                      scoring='r2',
                                                      display=display)
    
    datum = {
             "num_rows": X_train.shape[0],
             "num_features":X_train.shape[1],
             "test_r2": test_r2,
             "test_rmse": test_rmse,
             "train_r2": train_r2,
             "train_rmse": train_rmse,
             "y_pred": y_pred,
             "residuals": residuals,
             "cv_r2_mean": cv_r2_mean,
             "cv_r2_std": cv_r2_std,
             "cv_scores": cv_scores,
             "model": model}

    if groups is not None and len(groups.unique()) > 1:
        group_cv_r2_mean, group_cv_r2_std, group_cv_scores = \
            cross_validate_by_groups(cv_func,
                                     X_train,
                                     y_train_log,
                                     groups,
                                     n_splits=10, #groups.nunique(),
                                     scoring='r2',
                                     display=display)

        fold_sites, fold_plots = get_fold_sites(X_train, y_train_log, groups)
        
        group_info = {"group_cv_r2_mean": group_cv_r2_mean,
                      "group_cv_r2_std": group_cv_r2_std,
                      "group_cv_scores": group_cv_scores,
                      "fold_sites": fold_sites,
                      "fold_plots": fold_plots}

        datum.update(group_info)

    return datum

<a id="lin_reg"></a>
## LINEAR REGRESSION FUNCTIONS

In [38]:
def linear_reg_already_split(model_type, X_train, X_test,
                             y_train, y_test, cv, groups, label,display=True):
    return linear_reg(model_type,
                      X_train, X_test,
                      y_train, y_test,
                      cv, groups,
                      label, display=True)

In [39]:
def linear_reg_regular(model_type, X_var, y_var, cv, label,display=True):
    X_train, X_test, y_train, y_test = split_data_quantile(X_var, y_var)
    return linear_reg(model_type,
                      X_train, X_test,
                      y_train, y_test,
                      cv, None,
                      label,display=True)

In [40]:
# Group-aware linear regression with a single train/test split.
#
# Splits data using GroupShuffleSplit (80/20) ensuring no group appears
# in both train and test sets. Trains on the train split and evaluates
# on the held-out group(s). Grouped CV is performed on the training set
# to assess within-train generalization across groups.      
def linear_reg_groups(model_type, X_var, y_var, cv, groups,
                      label,display=True,
                      is_stratified = False):
    if is_stratified:
        X_train, X_test, y_train, y_test = split_data_plot_stratified(X_var, y_var, groups)
    else:
        X_train, X_test, y_train, y_test = split_data_by_groups(X_var, y_var, groups)
    
    # Split groups the same way as X and y
    groups_train = groups[y_train.index]
    groups_test  = groups[y_test.index]
    
    return linear_reg(model_type,
                      X_train, X_test,
                      y_train, y_test,
                      cv, groups_train,
                      label,display=True)

In [41]:
#Leave-One-Group-Out (LOGO) linear regression evaluation.
#
# Trains and evaluates a linear model by iterating over all unique groups,
# holding out one complete group as the test set each time. Only folds with
# at least 2 test samples are evaluated. Final test_r2 and test_rmse are
# the mean across all valid folds.
#
# How is this different compared to linear_reg_groups?
#  linear_reg_groups gives you one honest split
#  linear_reg_logo gives you N honest splits
def linear_reg_logo(model_type,
                    X_var, y_var,
                    cv, groups,
                    label, display=True):
    splits       = split_data_by_groups_logo(X_var, y_var, groups)
    fold_results = {}

    for X_train, X_test, y_train, y_test in splits:
        dataset_name = groups[y_test.index].iloc[0]
        print(f"  fold: {dataset_name}, y_test size: {len(y_test)}")  # ADD THIS
        if len(y_test) < 2:
            continue
        groups_train = groups[y_train.index]
        
        # Cap n_splits to number of unique training groups
        effective_cv = min(cv, len(groups_train.unique()))
        
        result = linear_reg(model_type,
                            X_train, X_test,
                            y_train, y_test,
                            effective_cv, groups_train,
                            label,
                            display)
        
        fold_results[dataset_name] = result

    # Aggregate fold results
    final                = fold_results[list(fold_results.keys())[-1]].copy()
    final['test_r2']     = np.mean([r['test_r2']   for r in fold_results.values()])
    final['test_rmse']   = np.mean([r['test_rmse'] for r in fold_results.values()])
    final['fold_sites']  = {k: v['test_r2'] for k, v in fold_results.items()}
    return final

In [42]:
def linear_reg(model_type,
               X_train, X_test, y_train, y_test,
               cv, groups, label, display=True):
    # Log-transform
    y_train_log = np.log1p(y_train)
    y_test_log  = np.log1p(y_test)
    
    # Scaling is necessary for linear regression regardless of whether
    # the target is log-transformed or not.
    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    if model_type == 'ridge':
        model    = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
        model_cv = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
    elif model_type == 'lasso':
        model    = LassoCV(alphas=[0.001, 0.01, 0.1, 1.0], cv=5)
        model_cv = LassoCV(alphas=[0.001, 0.01, 0.1, 1.0], cv=5)
    elif model_type == 'elasticnet':
        model    = ElasticNetCV(alphas=[0.001, 0.01, 0.1, 1.0],
                                l1_ratio=[0.2, 0.5, 0.8], cv=5)
        model_cv = ElasticNetCV(alphas=[0.001, 0.01, 0.1, 1.0],
                                l1_ratio=[0.2, 0.5, 0.8], cv=5)
    else:
        model    = LinearRegression()
        model_cv = LinearRegression()

    model.fit(X_train_scaled, y_train_log)

    """
    if display:
        plot_linear_reg_metrics(model_type, model, X_train_scaled, y_train_log, label)
    """
    return post_processing(model,
                           X_train_scaled, X_test_scaled,
                           y_train, y_test, y_train_log,
                           cv, model_cv, groups,
                           label,
                           display)

<a id="r_forest"></a>
## RANDOM FOREST FUNCTIONS

In [44]:
def randomForest_data_already_split(X_train, X_test,
                                    y_train, y_test,
                                    cv, groups,
                                    label, grid, display=True):
    return random_forest(X_train, X_test,
                         y_train, y_test,
                         cv, groups, label, grid)

In [45]:
def randomForest_regular(X_var, y_var, cv, label, grid, display=True):
    X_train, X_test, y_train, y_test = split_data_quantile(X_var, y_var)
    
    return random_forest(X_train, X_test,
                         y_train, y_test,
                         cv, None,
                         label, grid, display)

In [46]:
def randomForest_groups(X_var, y_var,
                        cv, groups,
                        label, grid, display=True,
                        is_stratified=False):
    if is_stratified:
        X_train, X_test, y_train, y_test = split_data_plot_stratified(X_var, y_var, groups)
    else:
        X_train, X_test, y_train, y_test = split_data_by_groups(X_var, y_var, groups)

    # Split groups the same way as X and y
    groups_train = groups[y_train.index]
    groups_test  = groups[y_test.index]
    
    return random_forest(X_train, X_test, y_train, y_test,
                         cv, groups_train,
                         label, grid, display)

In [47]:
def random_forest(X_train, X_test, y_train, y_test,
                  cv, groups, label, grid, display=True):
    X_var = pd.concat([X_train, X_test], axis=0)
    y_var = pd.concat([y_train, y_test], axis=0)

    if 'plot_id' in X_var.columns:
        X_var = X_var.drop(columns=['plot_id'])

    # Log-transform
    y_train_log = np.log1p(y_train)
    y_test_log  = np.log1p(y_test)
    
    # NOTE: Random forest does not need scaling.
    oob_errors    = None
    n_trees_range = None
    if grid:
        # Hyperparameter tuning
        param_grid = {
            'n_estimators'    : [100, 200, 500],
            'max_depth'       : [5, 10, 15, None],
            'min_samples_leaf': [2, 4, 8],
            'max_features'    : ['sqrt', 0.3, 0.5],
            'max_samples'     : [0.6, 0.8, 1.0]
        }
        grid_search = RandomizedSearchCV(RandomForestRegressor(random_state=42,
                                                               n_jobs=1), #-1
                                        param_grid,
                                        n_iter=80,
                                        cv=cv,
                                        scoring='r2',
                                        n_jobs=-1,
                                        random_state=42,
                                        verbose=1)
        grid_search.fit(X_train, y_train_log, groups=groups)
        print(f"Best params: {grid_search.best_params_}")
        
        # Train final model with best params
        rf_model = grid_search.best_estimator_
    
        # Cross-validation to confirm that the R^2 above is not by coincidence.
        rf_cv_model = RandomForestRegressor(**grid_search.best_params_,
                                            #n_estimators=500,
                                            random_state=42,
                                            n_jobs=1) #-1
    else:
        rf_model = RandomForestRegressor(
            n_estimators    = 500,
            max_features    = 'sqrt',
            min_samples_leaf= 2,
            oob_score       = True,   # enables OOB error tracking
            warm_start      = True,   # allows incremental fitting
            random_state    = 42,
            n_jobs          = 1 #-1
        )
        rf_cv_model = RandomForestRegressor(
            n_estimators    = 500,
            max_features    = 'sqrt',
            min_samples_leaf= 2,
            random_state    = 42,
            n_jobs          = 1 #-1
        )

        # OOB error curve — equivalent of a loss curve for RF
        #  The Out-of-Bag (OOB) error curve is a diagnostic
        #  metric useful to track the prediction error of a Random Forest model.
        #   - At each step, trees that were not used to train on a 
        #     particular sample vote on that sample's prediction — this
        #     is the out-of-bag estimate.
        #   - It is a built-in validation signal that requires no separate validation set.
        #   - As more trees are added the OOB error should decrease and then plateau.
        #   - If it never plateaus, 500 trees may not be enough.
        #   # If it plateaus early, you may not need 500 trees.

        n_trees_range = list(range(10, 501, 10))
        oob_errors    = []

        for n in n_trees_range:
            rf_model.n_estimators = n
            rf_model.fit(X_train, y_train_log)
            oob_errors.append(1 - rf_model.oob_score_)

        """
        if display:
            fig, ax = plt.subplots(figsize=(8, 4))
            ax.plot(n_trees_range, oob_errors, color='steelblue', linewidth=1.5)
            ax.axhline(y=min(oob_errors), color='red', linestyle='--',
                       linewidth=1, label=f'Min OOB error: {min(oob_errors):.4f}')
            ax.set_xlabel("Number of trees")
            ax.set_ylabel("OOB error  (1 − OOB R²)")
            ax.set_title(f"Random Forest OOB error — {label}")
            ax.legend()
            plt.tight_layout()
            plt.show()
            print(f"OOB R² at 500 trees : {rf_model.oob_score_:.4f}")
            print(f"Min OOB error       : {min(oob_errors):.4f} "
                  f"at {n_trees_range[oob_errors.index(min(oob_errors))]} trees")
        """
    
    datum = post_processing(rf_model,
                            X_train, X_test,
                            y_train, y_test, y_train_log,
                            cv, rf_cv_model, groups,
                            label,
                            display)
    
    importances = pd.Series(rf_model.feature_importances_, index=X_var.columns)
    imp_data    = {"importances"  : importances,
                   "oob_errors"   : oob_errors if not grid else None,
                   "n_trees_range": n_trees_range if not grid else None}
    datum.update(imp_data)

    return datum

<a id="xgboost"></a>
## XGBOOST FUNCTIONS

In [49]:
def xgboost_groups(X_var, y_var,
                   cv, groups,
                   label, grid, display=True,
                   is_stratified=False):
    if is_stratified:
        X_train, X_test, y_train, y_test = split_data_plot_stratified(X_var, y_var, groups)
    else:
        X_train, X_test, y_train, y_test = split_data_by_groups(X_var, y_var, groups)

    groups_train = groups[y_train.index]
    groups_test  = groups[y_test.index]

    return xgboost_model(X_train, X_test, y_train, y_test,
                         cv, groups_train,
                         label, grid, display)

In [51]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

def xgboost_model(X_train, X_test, y_train, y_test,
                  cv, groups, label, grid, display=True):

    X_var = pd.concat([X_train, X_test], axis=0)

    if 'plot_id' in X_var.columns:
        X_var = X_var.drop(columns=['plot_id'])

    # Log-transform target
    y_train_log = np.log1p(y_train)
    y_test_log  = np.log1p(y_test)

    if grid:
        param_grid = {
            'n_estimators'      : [100, 200, 500],
            'max_depth'         : [3, 5, 7],
            'learning_rate'     : [0.01, 0.05, 0.1, 0.2],
            'subsample'         : [0.6, 0.8, 1.0],
            'colsample_bytree'  : [0.5, 0.7, 1.0],
            'min_child_weight'  : [1, 5, 10],
            'reg_alpha'         : [0, 0.1, 1.0],   # L1
            'reg_lambda'        : [1, 5, 10],       # L2
        }
        grid_search = RandomizedSearchCV(
            XGBRegressor(random_state=42,
                         n_jobs=-1,
                         verbosity=0),
            param_grid,
            n_iter=80,
            cv=cv,
            scoring='r2',
            n_jobs=-1,
            random_state=42,
            verbose=1
        )
        grid_search.fit(X_train, y_train_log, groups=groups)
        print(f"Best params: {grid_search.best_params_}")

        xgb_model    = grid_search.best_estimator_
        xgb_cv_model = XGBRegressor(**grid_search.best_params_,
                                     random_state=42,
                                     n_jobs=-1,
                                     verbosity=0)
    else:
        xgb_model = XGBRegressor(
            n_estimators      = 500,
            max_depth         = 4,
            learning_rate     = 0.05,
            subsample         = 0.8,
            colsample_bytree  = 0.7,
            min_child_weight  = 5,
            reg_alpha         = 0.1,
            reg_lambda        = 5,
            random_state      = 42,
            early_stopping_rounds = 30,
            n_jobs            = -1,
            verbosity         = 0
        )
        xgb_cv_model = XGBRegressor(
            n_estimators      = 500,
            max_depth         = 4,
            learning_rate     = 0.05,
            subsample         = 0.8,
            colsample_bytree  = 0.7,
            min_child_weight  = 5,
            reg_alpha         = 0.1,
            reg_lambda        = 5,
            random_state      = 42,
            n_jobs            = -1,
            verbosity         = 0
        )
        xgb_model.fit(
            X_train, y_train_log,
            eval_set=[(X_test, y_test_log)],
            verbose=False
        )

    datum = post_processing(xgb_model,
                            X_train, X_test,
                            y_train, y_test, y_train_log,
                            cv, xgb_cv_model, groups,
                            "XGBOOST: " + label,
                            display)

    importances = pd.Series(xgb_model.feature_importances_, index=X_var.columns)
    datum.update({"importances": importances})

    return datum

<a id="lgbm"></a>
# Light GBM

In [54]:
def lightgbm_groups(X_var, y_var,
                    cv, groups,
                    label, grid, display=True,
                    is_stratified=False):
    if is_stratified:
        X_train, X_test, y_train, y_test = split_data_plot_stratified(X_var, y_var, groups)
    else:
        X_train, X_test, y_train, y_test = split_data_by_groups(X_var, y_var, groups)

    groups_train = groups[y_train.index]
    groups_test  = groups[y_test.index]

    return lightgbm_model(X_train, X_test, y_train, y_test,
                          cv, groups_train,
                          label, grid, display)

In [56]:
def lightgbm_model(X_train, X_test, y_train, y_test,
                   cv, groups, label, grid, display=True):

    X_var = pd.concat([X_train, X_test], axis=0)
    if 'plot_id' in X_var.columns:
        X_var = X_var.drop(columns=['plot_id'])

    y_train_log = np.log1p(y_train)
    y_test_log  = np.log1p(y_test)

    if grid:
        param_grid = {
            'n_estimators'     : [100, 200, 500],
            'max_depth'        : [3, 5, 7, -1],     # -1 means no limit
            'learning_rate'    : [0.01, 0.05, 0.1, 0.2],
            'num_leaves'       : [15, 31, 63],       # key LGBM parameter
            'min_child_samples': [10, 20, 50],       # min samples per leaf
            'subsample'        : [0.6, 0.8, 1.0],
            'colsample_bytree' : [0.5, 0.7, 1.0],
            'reg_alpha'        : [0, 0.1, 1.0],
            'reg_lambda'       : [1, 5, 10],
        }
        grid_search = RandomizedSearchCV(
            LGBMRegressor(random_state=42,
                          n_jobs=-1,
                          verbose=-1),
            param_grid,
            n_iter=80,
            cv=cv,
            scoring='r2',
            n_jobs=-1,
            random_state=42,
            verbose=1
        )
        grid_search.fit(X_train, y_train_log, groups=groups)
        print(f"Best params: {grid_search.best_params_}")

        lgbm_model    = grid_search.best_estimator_
        lgbm_cv_model = LGBMRegressor(**grid_search.best_params_,
                                       random_state=42,
                                       n_jobs=-1,
                                       verbose=-1)
    else:
        lgbm_model = LGBMRegressor(
            n_estimators      = 500,
            max_depth         = 4,
            learning_rate     = 0.05,
            num_leaves        = 15,
            min_child_samples = 20,
            subsample         = 0.8,
            colsample_bytree  = 0.7,
            reg_alpha         = 0.1,
            reg_lambda        = 5,
            random_state      = 42,
            n_jobs            = -1,
            verbose           = -1
        )
        lgbm_cv_model = LGBMRegressor(
            n_estimators      = 500,
            max_depth         = 4,
            learning_rate     = 0.05,
            num_leaves        = 15,
            min_child_samples = 20,
            subsample         = 0.8,
            colsample_bytree  = 0.7,
            reg_alpha         = 0.1,
            reg_lambda        = 5,
            random_state      = 42,
            n_jobs            = -1,
            verbose           = -1
        )
        lgbm_model.fit(X_train, y_train_log)

    datum = post_processing(lgbm_model,
                            X_train, X_test,
                            y_train, y_test, y_train_log,
                            cv, lgbm_cv_model, groups,
                            "LIGHTGBM: " + label,
                            display)

    raw_importances = lgbm_model.feature_importances_
    importances = pd.Series(
        raw_importances / raw_importances.sum(),  # normalise to 0-1
        index=X_var.columns
    )
    datum.update({"importances": importances})

    return datum

# MERF

In [59]:
def merf_groups(X_var, y_var,
                cv, groups, grid, label,
                display=True,
                is_stratified=False):
    if is_stratified:
        X_train, X_test, y_train, y_test = split_data_plot_stratified(X_var, y_var, groups)
    else:
        X_train, X_test, y_train, y_test = split_data_by_groups(X_var, y_var, groups)
    
    groups_train = groups[y_train.index]
    groups_test  = groups[y_test.index]
    
    return run_merf(X_train, X_test, y_train, y_test,
                    cv, groups_train, groups_test,
                    grid, label, display)

In [61]:
def run_merf(X_train, X_test, y_train, y_test,
             cv, groups_train, groups_test,
             grid, label, display=True):
    
    # Random intercept per plot
    Z_train = np.ones((len(X_train), 1))
    Z_test  = np.ones((len(X_test),  1))
    
    y_train_log = np.log1p(y_train)

    if grid:
        best_rf     = None
        best_score  = -np.inf
        grid_params = [
            (500, None, 2),
            (500, 10,   2),
            (200, 5,    4),
            (100, 5,    8),
        ]
        print("Grid search over MERF base RF params:")
        for n_estimators, max_depth, min_samples_leaf in grid_params:
            rf_candidate = RandomForestRegressor(
                n_estimators     = n_estimators,
                max_depth        = max_depth,
                min_samples_leaf = min_samples_leaf,
                random_state     = 42,
                n_jobs           = -1
            )
            merf_candidate = MERF(fixed_effects_model=rf_candidate,
                                  max_iterations=100,
                                  gll_early_stop_threshold=None)
            merf_candidate.fit(X_train, Z_train, groups_train, y_train_log)
            y_val  = merf_candidate.predict(X_train, Z_train, groups_train)
            score  = r2_score(y_train_log, y_val)
            print(f"  n_estimators={n_estimators}, max_depth={max_depth}, "
                  f"min_samples_leaf={min_samples_leaf} → Train R² (log)={score:.4f}")
            if score > best_score:
                best_score = score
                best_rf    = rf_candidate
        print(f"  Best train R² (log): {best_score:.4f}")
        rf_base = best_rf
    else:
        rf_base = RandomForestRegressor(n_estimators=500,
                                        random_state=42,
                                        n_jobs=-1)
    
    merf_model = MERF(fixed_effects_model=rf_base,
                      max_iterations=100,
                      gll_early_stop_threshold=None)

    print(f"\n MERF. {label}")
    print("MERF model fitting starts")
    merf_model.fit(X_train, Z_train, groups_train, y_train_log)
    print("MERF model fitting completed")
    
    # Predict
    y_pred_log       = merf_model.predict(X_test,  Z_test,  groups_test)
    y_train_pred_log = merf_model.predict(X_train, Z_train, groups_train)
    
    y_pred       = np.expm1(y_pred_log)
    y_train_pred = np.expm1(y_train_pred_log)
    
    test_r2   = r2_score(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    train_r2  = r2_score(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    
    train_r2_log = r2_score(y_train_log, y_train_pred_log)
    residuals    = y_test - y_pred

    # CV on log scale — matches post_processing convention
    print("MERF cross validation starts")
    start = time.perf_counter()
    cv_r2_mean, cv_r2_std, cv_scores = merf_cross_validate(X_train, y_train_log,
                                                           groups_train, rf_base,
                                                           n_splits=cv,
                                                           display=display)
    elapsed = time.perf_counter() - start
    print(f"Time taken for MERF cross validation: {elapsed:.4f}s")

    print("MERF Grouped cross validation starts")
    start = time.perf_counter()
    group_cv_r2_mean, group_cv_r2_std, \
        group_cv_scores, fold_sites, fold_plots = \
                merf_grouped_cross_validate(X_train, y_train_log,
                                            groups_train,
                                            rf_base,
                                            n_splits=10, # groups_train.nunique(),
                                            display=display)
    print(f"Time taken for MERF Grouped cross validation: {elapsed:.4f}s")
    
    if display:
        print(f"\n MERF: {label}")
        print(f"Test R²              : {test_r2:.4f}")
        print(f"Test RMSE            : {test_rmse:.2f} kg")
        print(f"Train R² (log scale) : {train_r2_log:.4f}")
        print(f"Train R² (orig scale): {train_r2:.4f}")
        print(f"Train RMSE           : {train_rmse:.2f} kg")
        
        print(f"\n Cross-validation ---")
        print(f"CV R² mean           : {cv_r2_mean:.4f}")
        print(f"CV R² std            : {cv_r2_std:.4f}")
        print(f"CV scores            : {np.round(cv_scores, 3)}")
        
        print(f"Num rows             : {len(X_train)}")
        print(f"Num groups           : {groups_train.nunique()}")
    
    importances = pd.Series(merf_model.fe_model.feature_importances_,
                            index=X_train.columns).sort_values(ascending=False)

    datum = {
        "num_features"      : len(X_train.columns),
        "test_r2"           : test_r2,
        "test_rmse"         : test_rmse,
        "train_r2"          : train_r2,
        "train_rmse"        : train_rmse,
        "y_pred"            : y_pred,
        "residuals"         : residuals,
        "cv_r2_mean"        : cv_r2_mean,
        "cv_r2_std"         : cv_r2_std,
        "cv_scores"         : cv_scores,
        "group_cv_r2_mean"  : group_cv_r2_mean,
        "group_cv_r2_std"   : group_cv_r2_std,
        "group_cv_scores"   : group_cv_scores,
        "fold_sites"        : fold_sites,
        "fold_plots"        : fold_plots,
        "importances"       : importances,
        "model"             : merf_model
    }

    return datum

In [63]:
# MERF cannot plug into sklearn's cross_validate directly. We need to implement it manually:

def merf_cross_validate(X_train, y_train_log, groups_train, rf_base, n_splits, display=True):
    from sklearn.model_selection import GroupKFold
    
    gkf = GroupKFold(n_splits=n_splits)
    scores = []
    
    for fold_train_idx, fold_val_idx in gkf.split(X_train, y_train_log, groups=groups_train):
        X_fold_train = X_train.iloc[fold_train_idx]
        X_fold_val   = X_train.iloc[fold_val_idx]
        y_fold_train = y_train_log.iloc[fold_train_idx]
        y_fold_val   = y_train_log.iloc[fold_val_idx]
        g_fold_train = groups_train.iloc[fold_train_idx]
        g_fold_val   = groups_train.iloc[fold_val_idx]
        
        Z_fold_train = np.ones((len(X_fold_train), 1))
        Z_fold_val   = np.ones((len(X_fold_val),   1))
        
        rf_fold   = clone(rf_base)   # fresh copy of best params
        
        merf_fold = MERF(fixed_effects_model=rf_fold,
                         max_iterations=30,
                         gll_early_stop_threshold=None)
        merf_fold.fit(X_fold_train, Z_fold_train,
                      g_fold_train, y_fold_train)
        
        y_fold_pred = merf_fold.predict(X_fold_val, Z_fold_val, g_fold_val)
        score = r2_score(y_fold_val, y_fold_pred)
        scores.append(score)
    
    scores = np.array(scores)
    if display:
        print(f"\n Cross-validation ---")
        print(f"CV R² mean: {scores.mean():.4f}")
        print(f"CV R² std : {scores.std():.4f}")
        print(f"CV scores : {np.round(scores, 3)}")
    return scores.mean(), scores.std(), scores

In [68]:
def merf_grouped_cross_validate(X_train, y_train_log, groups_train, rf_base, n_splits, display=True):
    from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
    from sklearn.base import clone
    
    if n_splits == groups_train.nunique():
        splitter = list(LeaveOneGroupOut().split(X_train, y_train_log, groups=groups_train))
    else:
        splitter = list(GroupKFold(n_splits=n_splits).split(X_train, y_train_log, groups=groups_train))
    
    total      = len(splitter)
    scores     = []
    fold_plots = []
    fold_sites = []
    
    for fold_idx, (fold_train_idx, fold_val_idx) in enumerate(splitter, 1):
        X_fold_train = X_train.iloc[fold_train_idx]
        X_fold_val   = X_train.iloc[fold_val_idx]
        y_fold_train = y_train_log.iloc[fold_train_idx]
        y_fold_val   = y_train_log.iloc[fold_val_idx]
        g_fold_train = groups_train.iloc[fold_train_idx]
        g_fold_val   = groups_train.iloc[fold_val_idx]
        
        plots_in_fold = g_fold_val.unique().tolist()
        site          = plots_in_fold[0].rsplit('_', maxsplit=1)[0]
        
        print(f"  [Grouped CV {fold_idx}/{total}] Fitting fold: {plots_in_fold} ...", end=' ', flush=True)
        
        Z_fold_train = np.ones((len(X_fold_train), 1))
        Z_fold_val   = np.ones((len(X_fold_val),   1))
        
        rf_fold   = clone(rf_base)
        merf_fold = MERF(fixed_effects_model=rf_fold,
                         max_iterations=100,
                         gll_early_stop_threshold=None)
        merf_fold.fit(X_fold_train, Z_fold_train, g_fold_train, y_fold_train)
        
        y_fold_pred = merf_fold.predict(X_fold_val, Z_fold_val, g_fold_val)
        score       = r2_score(y_fold_val, y_fold_pred)
        scores.append(score)
        fold_plots.append(plots_in_fold)
        fold_sites.append(site)
        
        print(f"R² = {score:.4f}")
    
    scores = np.array(scores)
    if display:
        print(f"\n Grouped Cross-validation ---")
        print(f"Grouped CV R² mean: {scores.mean():.4f}")
        print(f"Grouped CV R² std : {scores.std():.4f}")
        print(f"Grouped CV scores : {np.round(scores, 3)}")
    
    return scores.mean(), scores.std(), scores.tolist(), fold_sites, fold_plots

# TEST FRAMEWORK

In [ ]:
def filter_existing(feat_list, available):
    """Keep only features that survived filtering. Warn about dropped ones."""
    kept    = [f for f in feat_list if f in available]
    dropped = [f for f in feat_list if f not in available]
    if dropped:
        print(f"  Dropped (not in dataframe): {dropped}")
    return kept

def create_interact_terms(X, interact_col_names):
    for col in interact_col_names:
        if col in X.columns:
            X[f'height_x_{col}'] = X['height'] * X[col]

    return X

def generate_feature_combinations(feature_groups, categ_cols, required_groups=None):
    if required_groups is None:
        required_groups = []

    combos = []
    group_names = list(feature_groups.keys())

    for r in range(1, len(group_names) + 1):
        for selected_groups in combinations(group_names, r):

            # Skip useless combos
            if all(g in categ_cols for g in selected_groups):
                continue
            
            features = []
            for group in selected_groups:
                features.extend(feature_groups[group])

            features = list(dict.fromkeys(features))

            combos.append({
                "group_combo": selected_groups,
                "features": features,
                "num_features": len(features)
            })

    return combos

In [ ]:
#Consolidate EMIT band names into a single label for display
def summarize_features(features):
    if isinstance(features, str):
        import ast
        try:
            features = ast.literal_eval(features)
        except:
            return features

    emit_count = sum(1 for f in features if f.startswith('EMIT_'))
    other      = [f for f in features if not f.startswith('EMIT_')]

    summarized = []
    if emit_count > 0:
        summarized.append(f'EMIT bands ({emit_count})')
    summarized.extend(other)

    return str(summarized)

In [ ]:
class GROUP_INFO:
    def __init__(self, near_zero_sites, high_agb_sites,
                 near_zero_plots, high_agb_plots,
                 groups, cv):
        self.near_zero_sites = near_zero_sites
        self.high_agb_sites  = high_agb_sites
        self.near_zero_plots = near_zero_plots
        self.high_agb_plots  = high_agb_plots
        self.groups          = groups
        self.cv              = cv

In [ ]:
class BASE:
    def __init__(self, X_in, y_in, features, is_groups, group_info, is_stratified):
        self.y               = y_in
        self.features        = features
        self.near_zero_sites = group_info.near_zero_sites
        self.high_agb_sites  = group_info.high_agb_sites
        self.near_zero_plots = group_info.near_zero_plots
        self.high_agb_plots  = group_info.high_agb_plots
        self.groups          = group_info.groups
        self.cv              = group_info.cv
        self.is_groups       = is_groups
        self.is_stratified   = is_stratified
        self.results         = {}
        assert features
        self.X = X_in[features]
        cat_cols = get_categorical_cols(self.X)
        if cat_cols:
            self.X = pd.get_dummies(self.X, columns=cat_cols, dtype=int)

    def evaluate(self, label):
        if self.is_groups:
            verdict = evaluate_experiment(
                label            = label,
                results          = self.results,
                fold_sites       = self.results.get("fold_sites"),
                fold_plots       = self.results.get("fold_plots"),
                near_zero_sites  = self.near_zero_sites,
                high_agb_sites   = self.high_agb_sites,
                near_zero_plots  = self.near_zero_plots,
                high_agb_plots   = self.high_agb_plots
            )
            self.results["near_zero_sites"] = self.near_zero_sites
            self.results["high_agb_sites"]  = self.high_agb_sites
            self.results["near_zero_plots"] = self.near_zero_plots
            self.results["high_agb_plots"]  = self.high_agb_plots
        else:
            verdict = evaluate_experiment(
                label   = label,
                results = self.results
            )
        self.results["verdict"] = verdict

In [ ]:
class MODEL(BASE):
    MODEL_LIST = ["linear", "ridge", "lasso", "elasticnet"]  # static constant
    MODEL_NAMES = {
        "linear"     : "LINEAR REGRESSION",
        "ridge"      : "RIDGE REGRESSION",
        "lasso"      : "LASSO REGRESSION",
        "elasticnet" : "ELASTICNET REGRESSION",
        "rf"         : "RANDOM FOREST",
        "xgboost"    : "XGBOOST",
        "lgbm"       : "LightGBM",
        "merf"       : "MERF"
    }

    def __init__(self, X_in, y_in,
                 features,
                 is_groups, group_info,
                 is_stratified,
                 model_type,
                 is_grid=False):
        super().__init__(X_in, y_in, features, is_groups, group_info, is_stratified)
        self.model_type = model_type
        self.features   = features
        self.is_groups  = is_groups
        self.is_grid    = is_grid
        self.is_stratified = is_stratified
        self.group_info = group_info

    def evaluate(self, label):
        if self.is_groups:
            verdict = evaluate_experiment(
                label           = label,
                results         = self.results,
                fold_sites       = self.results.get("fold_sites"),
                fold_plots       = self.results.get("fold_plots"),
                near_zero_sites  = self.group_info.near_zero_sites,
                high_agb_sites   = self.group_info.high_agb_sites,
                near_zero_plots  = self.group_info.near_zero_plots,
                high_agb_plots   = self.group_info.high_agb_plots
            )
        else:
            verdict = evaluate_experiment(
                label   = label,
                results = self.results
            )

        self.results["model"]          = str(self.MODEL_NAMES[self.model_type])
        self.results["features"]       = summarize_features(self.features)
        self.results["grouping"]       = "Yes" if self.is_groups else "No"
        self.results["stratified"]     = "Yes" if self.is_stratified else "No"
        self.results["hyperparameter"] = "Yes" if self.is_grid else "No"

        self.results["near_zero_sites"]= self.group_info.near_zero_sites
        self.results["high_agb_sites"] = self.group_info.high_agb_sites
        self.results["near_zero_plots"]= self.group_info.near_zero_plots
        self.results["high_agb_plots"] = self.group_info.high_agb_plots
        self.results["fold_sites"]     = self.results.get("fold_sites")
        self.results["fold_plots"]     = self.results.get("fold_plots")
        
        self.results["verdict"] = verdict
    
    def run(self, label):
        label += f"Model: {self.MODEL_NAMES[self.model_type]}"
        label += f", Grouping? {"Yes" if self.is_groups else "No"}"
        label += f", Hypertuned? {"Yes" if self.is_grid else "No"}"
        label += f", Features: {summarize_features(self.features)}"

        if self.model_type in ('linear', 'ridge', 'lasso', 'elasticnet'):
            if self.is_groups:
                self.results = linear_reg_groups(self.model_type, self.X, self.y,
                                                 self.cv, self.groups, label,
                                                 is_stratified=self.is_stratified)
            else:
                self.results = linear_reg_regular(self.model_type, self.X, self.y,
                                                  self.cv, label)

        elif self.model_type == 'rf':
            if self.is_groups:
                self.results = randomForest_groups(self.X, self.y, self.cv,
                                                   self.groups, label,
                                                   grid=self.is_grid, display=True,
                                                   is_stratified=self.is_stratified)
            else:
                self.results = randomForest_regular(self.X, self.y, self.cv,
                                                    label, grid=self.is_grid,
                                                    display=True)

        elif self.model_type == 'xgboost':
            assert self.is_groups
            self.results = xgboost_groups(self.X, self.y, self.cv, self.groups,
                                          label, grid=self.is_grid, display=True,
                                          is_stratified=self.is_stratified)

        elif self.model_type == 'lgbm':
            assert self.is_groups
            self.results = lightgbm_groups(self.X, self.y, self.cv, self.groups,
                                           label, grid=self.is_grid, display=True,
                                           is_stratified=self.is_stratified)

        elif self.model_type == 'merf':
            assert self.is_groups
            self.results = merf_groups(self.X, self.y, self.cv, self.groups,
                                       self.is_grid, label, display=True,
                                       is_stratified=self.is_stratified)
        else:
            raise ValueError(f"Unknown model_type: {self.model_type}")

        self.evaluate(label)
        if self.model_type not in ('linear', 'ridge', 'lasso', 'elasticnet'):
            show_importances(self.results)
        return label

In [ ]:
def run_experiment(X, y,
                   is_groups, group_info,
                   features_list,
                   model_type,
                   label="",
                   is_grid=False,
                   is_stratified=False,
                   exp_total_list=None,
                   exp_by_category=None,
                   linear_variants=None):
    experiment = {}

    if model_type == 'linear':
        assert linear_variants is not None
        combos = [(f, m) for f in features_list for m in linear_variants]
    else:
        combos = [(f, model_type) for f in features_list]

    total = len(combos)
    for i, (features, mtype) in enumerate(combos, 1):
        print(f"\n[{i}/{total}]")
        model = MODEL(X, y,
                      features,
                      is_groups, group_info,
                      is_stratified,
                      model_type=mtype,
                      is_grid=is_grid)
        
        new_label = f"EXPERIMENT-{i}. {label},"
        new_label = model.run(new_label)
        exp_by_category[new_label] = model.results

        global_label = f"{label}, " + new_label
        exp_total_list[global_label] = model.results

    print(f"\nCompleted: {total}/{total} experiments")

In [ ]:
#Filter experiment results to find the best models.
# Priority 1: ACCEPTABLE verdicts
# Priority 2: MARGINAL verdicts that pass relaxed quality checks
def filter_best_experiments(experiments: dict,
                            grouped_only=False,
                            cv_test_gap_max: float = 0.10,
                            grouped_cv_min:  float = 0.15) -> list:
    filtered = {}
    for label, res in experiments.items():
        verdict       = res.get("verdict", "")
        test_r2       = res.get("test_r2")
        train_r2      = res.get("train_r2")
        cv_mean       = res.get("cv_r2_mean")
        group_cv_mean = res.get("group_cv_r2_mean")
        is_grouped    = group_cv_mean is not None
        primary_cv    = group_cv_mean if is_grouped else cv_mean

        if grouped_only and not is_grouped:
            continue

        is_acceptable = "\u2705" in verdict
        is_marginal   = "\u26a0\ufe0f" in verdict or "MARGINAL" in verdict

        if is_acceptable:
            filtered[label] = res
            continue

        if is_marginal and test_r2 is not None and primary_cv is not None:
            cv_test_gap    = abs(primary_cv - test_r2)
            train_test_gap = (train_r2 - test_r2) if train_r2 is not None else 999
            cv_above_min   = (group_cv_mean >= grouped_cv_min) if is_grouped else (cv_mean >= 0)
            if all([cv_above_min,
                    cv_test_gap    <= cv_test_gap_max,
                    train_test_gap <= 0.15,
                    test_r2        > 0]):
                filtered[label] = res

    if not filtered:
        print("No acceptable or marginal-pass experiments found.")
        print("Sample verdicts:", [v.get('verdict') for v in list(experiments.values())[:3]])
        return []

    def sort_key(item):
        _, res        = item
        is_acceptable = "\u2705" in res.get("verdict", "")
        group_cv_mean = res.get("group_cv_r2_mean")
        primary_cv    = group_cv_mean if group_cv_mean is not None else res.get("cv_r2_mean", 0)
        test_r2       = res.get("test_r2", 0)
        return (int(is_acceptable), primary_cv, test_r2)

    sorted_items = sorted(filtered.items(), key=sort_key, reverse=True)
    print(f"Found {len(sorted_items)} experiments — best first.")
    return sorted_items

In [ ]:
def print_experiment(results):
    yes_no = lambda x: "Yes" if x else "No"
    
    print(f"\n EXPERIMENT")
    print(f"  Model                : {results.get('model')}")
    print(f"  Features             : {results.get('features')}")
    print(f"  Num Features         : {results['num_features']}")
    print(f"  Grouping             : {results.get('grouping')}")
    print(f"  Stratified split     : {results.get('stratified')}")
    print(f"  Hyperparameter tuned : {results.get('hyperparameter')}")
    print(f"  Test R²              : {results['test_r2']:.4f}")
    print(f"  Test RMSE            : {results['test_rmse']:.2f} kg")
    print(f"  Train R² (log scale) : {results.get('train_r2_log', float('nan')):.4f}")
    print(f"  Train R² (orig scale): {results['train_r2']:.4f}")
    print(f"  Train RMSE           : {results['train_rmse']:.2f} kg")

    print(f"\n Cross-validation ---")
    print(f"  CV R² mean: {results['cv_r2_mean']:.4f}")
    print(f"  CV R² std : {results['cv_r2_std']:.4f}")
    print(f"  CV scores : {np.round(results['cv_scores'], 3)}")

    if results.get('group_cv_r2_mean') is not None:
        print(f"\n Grouped Cross-validation ---")
        print(f"  Grouped CV R² mean : {results['group_cv_r2_mean']:.4f}")
        print(f"  Grouped CV R² std  : {results['group_cv_r2_std']:.4f}")
        print(f"  Grouped CV scores  : {np.round(results['group_cv_scores'], 3)}")
        print(f"\n  Fold plots       : {results['fold_plots']}")

        print(f"\n  near_zero_sitesplots: {results['near_zero_sites']}")
        print(f"\n  high_agb_sites      : {results['high_agb_sites']}")
        print(f"\n  near_zero_plots     : {results['near_zero_plots']}")
        print(f"\n  high_agb_plots      : {results['high_agb_plots']}")
        
    print(f"\n Verdict: {results.get('verdict', 'N/A')}")

In [ ]:
def print_experiment_from_row(row):
    print(f"\nExperiment")
    print(f"  Model               : {row.get('model')}")
    print(f"  Features            : {row.get('features')}")
    print(f"  Num Features        : {row['Num_features']}")
    print(f"  Grouping            : {row.get('grouping')}")
    print(f"  Stratified split    : {row.get('stratified')}")
    print(f"  Hyperparameter tuned: {row.get('hyperparameter')}")
    
    print(f"  Test R²     : {row['Test R²']:.4f}")
    print(f"  Test RMSE   : {row['Test RMSE']:.2f} kg")
    print(f"  Train R²    : {row['Train R²']:.4f}")
    print(f"  Train RMSE  : {row['Train RMSE']:.2f} kg")

    print(f"\n Cross-validation ---")
    print(f"  CV R² mean: {row['CV R² Mean']:.4f}")
    print(f"  CV R² std : {row['CV R² Std']:.4f}")
    print(f"  CV scores : {np.round(row['cv_scores'], 3)}")

    if row.get('Group CV R² Mean') is not None and row['Group CV R² Mean'] != '':
        print(f"\nGrouped Cross-validation ---")
        print(f"  Grouped CV R² mean : {row['Group CV R² Mean']:.4f}")
        print(f"  Grouped CV R² std  : {row['Group CV R² Std']:.4f}")
        print(f"  Grouped CV scores  : {np.round(row['group_cv_scores'], 3)}")
        print(f"  Grouped CV-Test Gap: {row['Group CV-Test Gap']:.4f}")
        print(f"  -ve folds          : {row['-ve Folds']}")
        
    print(f"\n  Verdict: {row['verdict']}")

In [ ]:
def tabulate_results(results, acceptable_only=False, grouped_only=False, top_n=10):
    if not len(results):
        return
    
    rows = []
    
    # works for both list of tuples and dict
    items = results if isinstance(results, list) else results.items()
    for label, metrics in items:        
        group_cv_mean = metrics.get('group_cv_r2_mean')
        has_group_cv  = group_cv_mean is not None
        
        # Skip non-grouped results if grouped_only is set
        if grouped_only and not has_group_cv:
            continue

        row = {
            'model'       : metrics.get('model'),
            #'Num_features': round(metrics['num_features'], 4),
            'Features'    : metrics.get('features'),
            'Groups?'     : metrics.get('grouping'),
            #'stratified'  : metrics.get('stratified'),
            'Tuned?'     : metrics.get('hyperparameter'),
            'Test R²'    : round(metrics['test_r2'], 4),
            'Test RMSE'  : round(metrics['test_rmse'], 2),
            'Train R²'   : round(metrics['train_r2'], 4),
            'Train RMSE' : round(metrics['train_rmse'], 2),
            'CV R² Mean' : round(metrics['cv_r2_mean'], 4),
            'CV R² Std'  : round(metrics['cv_r2_std'], 4),
            'CV scores'  : np.round(metrics['cv_scores'], 3).tolist()
        }
        if has_group_cv:
            group_cv_mean = metrics.get('group_cv_r2_mean', float('nan'))
            group_cv_scores = metrics.get('group_cv_scores', [])
            test_r2       = metrics['test_r2']
            cv_test_gap   = abs(group_cv_mean - test_r2) if group_cv_mean is not None else float('nan')

            # Count negative folds and list which plots scored negative
            if group_cv_scores is not None and len(group_cv_scores) > 0:
                fold_sites      = metrics.get('fold_sites', [])
                negative_folds  = [
                    fold_sites[i] if i < len(fold_sites) else f"fold_{i}"
                    for i, s in enumerate(group_cv_scores) if s < 0
                ]
                n_negative      = len(negative_folds)
                negative_label  = f"{n_negative} ({', '.join(negative_folds)})" if n_negative > 0 else "none"
            else:
                n_negative     = 0
                negative_label = "none"
            
            group_cv_r2_mean  = round(group_cv_mean, 4)
            group_cv_r2_std   = round(metrics.get('group_cv_r2_std', float('nan')), 4)
            negative_folds    = negative_label
            group_cv_test_gap = round(cv_test_gap, 4)
            group_cv_scores   = np.round(metrics['group_cv_scores'], 3).tolist()
        else:
            group_cv_r2_mean  = None
            group_cv_r2_std   = None
            group_cv_test_gap = None
            group_cv_scores   = None
            negative_folds    = None

        row['Group CV R² Mean'] = group_cv_r2_mean
        row['Group CV R² Std']  = group_cv_r2_std
        row['Group CV-Test Gap']= group_cv_test_gap
        row['-ve Folds']  = negative_folds
        row['Group CV scores']  = group_cv_scores
        row['verdict']          = metrics['verdict']
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    # Check globally whether any row has group CV
    any_group_cv = df['Group CV R² Mean'].apply(lambda x: x != 'N/A').any()
    sort_col     = 'Group CV R² Mean' if any_group_cv else 'CV R² Mean'

    df = df.sort_values(sort_col, ascending=False).reset_index(drop=True)
    if top_n is not None:
        df = df.head(top_n)
    
    if acceptable_only:
        df = df[df['verdict'].str.contains('ACCEPTABLE|MARGINAL', na=False)].reset_index(drop=True)
        print(f"Showing {len(df)} acceptable experiments")

    if df.empty:
        print("No acceptable experiments found.")
        return
    
    highlight_max_cols = ['Test R²', sort_col]
    highlight_min_cols = ['Test RMSE']
    if any_group_cv:
        highlight_min_cols.append('Group CV R² Std')
        highlight_min_cols.append('Group CV-Test Gap')   # smaller gap is better

    def safe_fmt(x):
        try:
            return f'{x:.4g}'
        except (TypeError, ValueError):
            return '' if x is None else x

    styled = (
        df.style
        .format(safe_fmt)
        .highlight_max(subset=highlight_max_cols, color='lightgreen')
        .highlight_min(subset=highlight_min_cols, color='lightgreen')
    )
    display(styled)

    return df

In [ ]:
def evaluate_experiment(label, results,
                        fold_sites=None,
                        fold_plots=None,
                        near_zero_sites=None,
                        high_agb_sites=None,
                        near_zero_plots=None,
                        high_agb_plots=None):

    near_zero_sites = near_zero_sites or []
    high_agb_sites  = high_agb_sites  or []
    near_zero_plots = near_zero_plots or []
    high_agb_plots  = high_agb_plots  or []

    test_r2    = results.get('test_r2')
    test_rmse  = results.get('test_rmse')
    train_r2   = results.get('train_r2')
    cv_mean    = results.get('cv_r2_mean')
    cv_std     = results.get('cv_r2_std')
    cv_scores  = results.get('cv_scores')

    group_cv_mean   = results.get('group_cv_r2_mean')
    group_cv_std    = results.get('group_cv_r2_std')
    group_cv_scores = results.get('group_cv_scores')

    has_cv      = cv_mean is not None and cv_scores is not None
    is_grouped  = group_cv_scores is not None and fold_sites is not None

    # ── Thresholds ────────────────────────────────────────────────
    GROUPED_CV_MEAN_MIN  = 0.15
    GROUPED_CV_STD_MAX   = 0.25
    CV_TEST_GAP_MAX      = 0.05
    TRAIN_TEST_GAP_MAX   = 0.10

    print("=" * 60)
    print(f" EVALUATION: {label}")
    print("=" * 60)

    print(f"\nTest set:")
    print(f"  R²   : {test_r2:.3f}")
    print(f"  RMSE : {test_rmse:.2f} kg")

    checks = []

    # Check 1 — test R² positive
    if test_r2 > 0:
        print(f"\n {GREEN_TICK} Test R² is positive ({test_r2:.3f})")
        checks.append(True)
    else:
        print(f"\n {RED_CROSS} Test R² is negative ({test_r2:.3f})")
        checks.append(False)

    # Check 2 — regular CV (skipped for MERF)
    if has_cv:
        print(f"\nRegular CrossValidation:")
        print(f"  Mean   : {cv_mean:.3f}")
        print(f"  Std    : {cv_std:.3f}")
        print(f"  Scores : {np.round(cv_scores, 3)}")

        if cv_mean > 0:
            print(f" {GREEN_TICK} CV mean is positive ({cv_mean:.3f})")
            checks.append(True)
        else:
            print(f" {RED_CROSS} CV mean is negative ({cv_mean:.3f})")
            checks.append(False)
    else:
        print(f"\nRegular CrossValidation: N/A (MERF)")

    if is_grouped:
        scores = np.array(group_cv_scores)

        print(f"\nGrouped CrossValidation:")
        for i, (site, score) in enumerate(zip(fold_sites, scores)):
            if score < 0:
                contaminated = fold_plots is not None and any(
                    p != site and p in high_agb_plots
                    for p in fold_plots[i]
                )
                if contaminated:
                    contam_plot = next(
                        p for p in fold_plots[i]
                        if p != site and p in high_agb_plots
                    )
                    reason = f"{WARNING}  fold contamination — {contam_plot} in same fold"
                elif site in high_agb_plots:
                    reason = f"{RED_CROSS} genuine generalization failure"
                elif site in near_zero_plots:
                    reason = f"{WARNING}  near-zero variance artifact"
                elif get_site_from_plot(site) in near_zero_sites:
                    reason = f"{WARNING}  near-zero variance artifact"
                else:
                    reason = f"{RED_CROSS} negative — cause unknown"
            else:
                reason = f"{GREEN_TICK}"
            print(f"  {site:20s} : {score:>7.3f}  {reason}")

        print(f"\n  Mean : {group_cv_mean:.3f}")
        print(f"  Std  : {group_cv_std:.3f}")

        # Check 3 — grouped CV mean
        if group_cv_mean >= GROUPED_CV_MEAN_MIN:
            print(f"\n  {GREEN_TICK} Grouped CV mean above {GROUPED_CV_MEAN_MIN} ({group_cv_mean:.3f})")
            checks.append(True)
        else:
            print(f"\n  {RED_CROSS} Grouped CV mean below {GROUPED_CV_MEAN_MIN} ({group_cv_mean:.3f})")
            checks.append(False)

        # Check 4 — grouped CV std
        if group_cv_std <= GROUPED_CV_STD_MAX:
            print(f"  {GREEN_TICK} Grouped CV std below {GROUPED_CV_STD_MAX} ({group_cv_std:.3f})")
            checks.append(True)
        else:
            print(f"  {WARNING}  Grouped CV std above {GROUPED_CV_STD_MAX} ({group_cv_std:.3f}) — high variance across folds")
            checks.append(False)

        # Check 5 — unexplained negatives
        unexplained = []
        for i, (site, score) in enumerate(zip(fold_sites, scores)):
            if score >= 0:
                continue
            site_name    = get_site_from_plot(site)
            contaminated = fold_plots is not None and any(
                p != site and p in high_agb_plots
                for p in fold_plots[i]
            )
            if (not contaminated
                    and site not in near_zero_plots
                    and site_name not in near_zero_sites
                    and site not in high_agb_plots):
                unexplained.append(site)

        if not unexplained:
            print(f"  {GREEN_TICK} All negative folds explained")
            checks.append(True)
        else:
            print(f"  {RED_CROSS} Unexplained negative folds: {unexplained}")
            checks.append(False)

        # Check 6 — high AGB sites
        high_agb_failures = [
            site for site, score in zip(fold_sites, scores)
            if (get_site_from_plot(site) in high_agb_sites
                or site in high_agb_plots)
            and score < 0
            and site not in near_zero_plots
            and get_site_from_plot(site) not in near_zero_sites
        ]
        if not high_agb_failures:
            print(f"  {GREEN_TICK} High-AGB sites generalize correctly")
            checks.append(True)
        else:
            print(f"  {RED_CROSS} High-AGB sites failed: {high_agb_failures}")
            checks.append(False)

        # Check 7 — CV/test gap
        cv_test_gap = abs(group_cv_mean - test_r2)
        if cv_test_gap <= CV_TEST_GAP_MAX:
            print(f"  {GREEN_TICK} Grouped CV and test R² agree (gap: {cv_test_gap:.3f})")
            checks.append(True)
        else:
            print(f"  {WARNING}  Grouped CV and test R² diverge (gap: {cv_test_gap:.3f} > {CV_TEST_GAP_MAX}) — possible leakage or instability")
            checks.append(False)

    # Check 8 — train/test gap (always applies)
    if train_r2 is not None:
        train_test_gap = train_r2 - test_r2
        if train_test_gap <= TRAIN_TEST_GAP_MAX:
            print(f"  {GREEN_TICK} Train/test R² gap acceptable (gap: {train_test_gap:.3f})")
            checks.append(True)
        else:
            print(f"  {WARNING}  Train R² well above test R² (gap: {train_test_gap:.3f} > {TRAIN_TEST_GAP_MAX}) — overfitting signal")
            checks.append(False)

    print(f"\n{'─' * 60}")
    grouped_cv_clean = is_grouped and all(checks[2:])

    if all(checks):
        print(f"VERDICT: {GREEN_TICK} ACCEPTABLE")
        verdict = f"{GREEN_TICK} ACCEPTABLE"
    elif grouped_cv_clean and has_cv and not checks[1]:
        # ── FIX: negative regular CV is a warning — demote to MARGINAL ──
        print(f"VERDICT: {WARNING} MARGINAL (grouped CV clean but regular CV negative)")
        verdict = f"{WARNING} MARGINAL (grouped CV clean but regular CV negative)"
    elif checks[0] and (not has_cv or checks[1]):
        print(f"VERDICT: {WARNING}  MARGINAL — positive metrics but concerns")
        verdict = f"{WARNING}  MARGINAL — positive metrics but concerns"
    else:
        print(f"VERDICT: {RED_CROSS} REJECTED")
        verdict = f"{RED_CROSS} REJECTED"

    return verdict

**COMMENTS**  
**Case 1**  
 - Regular CV -ve, Grouped CV: X
 - The model generalises well when plot leakage is prevented (grouped CV), but fails when plots bleed across folds (regular CV).
 - This suggests the model has learned some plot-level signal that does not fully generalise.
 - The negative regular CV is a real warning.

**Case 2**  
 - Regular CV +ve, Grouped CV: slightly lower than X
 - Both evaluation methods agree.
 - The model generalises consistently regardless of how folds are constructed.
 - This is more trustworthy.

**Which one is preferrable?**  
 - Case 2 is more preferable.
 - Consistency between regular and grouped CV is a stronger signal than grouped CV alone.